In [0]:
# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="lit-scope-2", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="lit-scope-2", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="lit-scope-2", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="lit-scope-2", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
print("hello")

In [0]:
import xarray
print(xarray.__version__)

In [0]:
import xarray as xr, numpy as np, math
from pyspark import SparkFiles

raw = "gs://lit-denser-data/raw/"
output = {"pr": "pr_2020.nc", "tmmx": "tmmx_2020.nc", "rmin": "rmin_2020.nc"}

for f in output.values():
    sc.addFile(f"{raw}/{f}") # copeis each file to every worker

ds = xr.open_dataset(SparkFiles.get("pr_2020.nc"))
n_days = ds.sizes["day"]
n_days

In [0]:
def read(n):
    data = {}
    for i in ["pr","tmmx","rmin"]:
        df = xr.open_dataset(SparkFiles.get(output[i]))
        ls = list(df.data_vars)[0] 
        data[i] = df[ls].isel(day=n) 
        
    table = xr.Dataset(data).to_dataframe().reset_index() # reset_index changes lat lon to columns from labels
    table["day"] = table["day"].astype(str).str[:10] 

    return table[["day","lat","lon","pr","tmmx","rmin"]].itertuples(index=False)
met = sc.parallelize(range(n_days), 24).flatMap(read)
met



In [0]:
met.take(2)

In [0]:
def clean(row):
    date,lat,lon,pr,tmmx,rmin = row
    for v in [pr,tmmx,rmin]:
        if math.isnan(v):
            return []
    tmmx = tmmx - 273.15
    return [(date, round(float(lat), 4), round(float(lon), 4), float(pr), float(tmmx), float(rmin))]
met_clean = met.flatMap(clean)
met_clean.take(5)


In [0]:
met_clean.map(lambda x: ",".join(map(str, x))) \
    .saveAsTextFile("gs://lit-denser-data/cleaned/gridmet")

In [0]:
import csv
from datetime import datetime

aq_raw = sc.textFile("gs://lit-denser-data/raw/daily_42101_2020.csv,"
                     "gs://lit-denser-data/raw/daily_88101_2020.csv").mapPartitions(csv.reader)
header = aq_raw.first()
col = {name: i for i, name in enumerate(header)}
row_length = len(header)
header

In [0]:
data = aq_raw.filter(lambda r: r[0] != "State Code")

for k, v in sorted(data.map(lambda r: (r[col["Parameter Code"]], r[col["Sample Duration"]])).countByValue().items()):\
    print(k, v)

In [0]:
Names = {"42101": "CO", "88101": "PM25"}
Keep = {"42101": {'8-HR RUN AVG END HOUR'},"88101":{'24-HR BLK AVG'}}
Values = {"42101": "1st Max Value", "88101": "Arithmetic Mean"}


In [0]:
def clean(row):
    if len(row) != row_length or row[0] == 'State Code':
        return []
    if row[col["Parameter Code"]] not in Keep or row[col["Sample Duration"]] not in Keep[row[col["Parameter Code"]]]:
        return []
    if row[col['Event Type']] == 'Excluded':
        return []
    lat = float(row[col["Latitude"]])
    lon = float(row[col["Longitude"]])
    aqi = int(row[col["AQI"]])
    value = float(row[col[Values[row[col["Parameter Code"]]]]])

    date = row[col["Date Local"]].strip()
    poc = row[col["POC"]].strip()
    state = row[col["State Name"]].strip()

    for i in [date,poc,state]:
        if i == '':
            return []

    return [(date, round(lat, 4), round(lon, 4), aqi,Names[row[col["Parameter Code"]]], value, poc, state)]
aq_clean = aq_raw.flatMap(clean).distinct()
aq_clean.take(5)

    

In [0]:
aq_clean.map(lambda r: "\t".join(map(str, r))).saveAsTextFile("gs://lit-denser-data/cleaned/air_quality")

In [0]:
aq = sc.textFile("gs://lit-denser-data/cleaned/air_quality")
print(aq.count())
aq.take(3)